# Fixes after all Curation steps 

After following all the curation scripts, I noticed, that a few curated models did not show growth on complete medium. This script provides investigation about the problems and fixes to these models. 

#### Imports 

In [3]:
import cobra
import json
import pandas as pd
import re
import os
from cobra.io import read_sbml_model, write_sbml_model
from cobra import Reaction, Metabolite
from collections import defaultdict


In [4]:
working_dir = '/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models'
draft_model_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/"
model_dir = os.path.join(working_dir, "mb1_mdr_rdr_dp_mb2_lib_bz")
mb_dir = os.path.join(working_dir, "mb1/")

#### Check production of biomass precursors 

In [5]:
# taken from Lisa 
def add_new_rxn(model, id, name, lb, ub, stoich):
    if id not in model.reactions:
        new_rxn = Reaction(id=id, name=name, lower_bound=lb, upper_bound=ub)

        met_objs = {}
        for met_id, coeff in stoich.items():
            met_obj = model.metabolites.get_by_id(met_id) if met_id in model.metabolites else None
            if met_obj is None:
                return f"{met_id} not in {model.id}; {id} was not added"
            met_objs[met_obj] = coeff

        new_rxn.add_metabolites(met_objs)
        model.add_reactions([new_rxn])
        return None
    else:
        return f"{id} already in {model.id}"
    
def add_new_met(model, id, name, formula, charge, compartment):
    if id not in model.metabolites:
        new_met = Metabolite(id, name=name, formula=formula, charge=charge, compartment=compartment)
        model.add_metabolites([new_met])
        return None
    else:
        return f"{id} already in {model.id}"


In [6]:
def check_growth(model):
    biomass_test = []
    blocked_reactions = {}  # dict to collect blocked reactions per model

    for met in model.reactions.Growth.reactants:
        # temporary demand reaction for this specific metabolite: "metabolite -> " (draining it from the system)
        with model as temporary_model:
            try:
                demand_rxn = temporary_model.add_boundary(met, type="demand")
                temporary_model.objective = demand_rxn
                solution = temporary_model.optimize()
                status = "Pass" if solution.objective_value > 1e-5 else "BLOCKED"
                biomass_test.append(
                    {
                        "Metabolite_ID": met.id,
                        "Metabolite_Name": met.name,
                        "Production_Flux": solution.objective_value,
                        "Status": status,
                    }
                )
            except Exception as e:
                status = f"Error: {str(e)}"
                biomass_test.append(
                    {
                        "Metabolite_ID": met.id,
                        "Metabolite_Name": met.name,
                        "Production_Flux": 0.0,
                        "Status": status,
                    }
                )

    df_test = pd.DataFrame(biomass_test)
    blocked_components = df_test[df_test["Status"] == "BLOCKED"]

    # collect blocked reactions for this model
    if not blocked_components.empty:
        blocked_reactions[model.id] = blocked_components["Metabolite_ID"].tolist()
    else:
        blocked_reactions[model.id] = []

    print(f"Found {len(blocked_components)} biomass components that cannot be synthesized for model {model.id}.")
    return blocked_components, blocked_reactions

In [7]:
def demand_reaction(model, list_problem_met_ids): #create demand reaction for metabolite to check whether it can ever be produced
    demand_flux_d = {}
    for met_id in list_problem_met_ids:
        with model as temp_model:
            try:
                met_obj = temp_model.metabolites.get_by_id(met_id)
                
                # isolated demand reaction
                demand_rxn = temp_model.add_boundary(met_obj, type="demand") #"A demand reaction is an irreversible reaction that consumes an intracellular metabolite"
                temp_model.objective = demand_rxn
                #print(model.demands)
                sol = temp_model.optimize()
                
                #print(f"Max production flux of {met_id:10}: {sol.objective_value:.4f}")
                demand_flux_d[met_id] = sol.objective_value
            except KeyError:
                print(f"Error: Metabolite '{met_id}' not found in the model.")
            except Exception as e:
                print(f"Error optimizing {met_id}: {e}")
    return demand_flux_d

In [8]:
def get_non_growing_mets(demand2flux_dict): #creates list of metabolites that can not be produced after demand reaction checks
    non_growing_mets = []
    for met_id,flux in demand2flux_dict.items():
        if flux == 0.000:
            non_growing_mets.append(met_id)
    return non_growing_mets

In [9]:
def artificial_cytoplasm_addition(model, met_id): #artificially add metabolites to cytosol 
    if met_id.endswith("_c"):
        clean_met_id = met_id[:-2]
    elif met_id.endswith("_p"):
        clean_met_id = met_id[:-2]
    else:
        clean_met_id = met_id

    cyto_met_id = clean_met_id + "_c"
    reac_id_str = met_id + "_syn"
    
    reac = Reaction(reac_id_str)
    reac.name = f"Artificial synthesis of {clean_met_id}"
    reac.subsystem = "Synthetic"
    reac.lower_bound = 0.0  
    reac.upper_bound = 1000.0
    if cyto_met_id in model.metabolites:
        met_to_add = model.metabolites.get_by_id(cyto_met_id)
    else:
        met_to_add = Metabolite(
            id=cyto_met_id,
            name=f"{clean_met_id} (cytoplasm)",
            compartment="c"
        )
        model.add_metabolites([met_to_add])
        
    reac.add_metabolites({met_to_add: 5.0})
    return reac

In [10]:
def test_biomass_prec(model): #test all biomass precursors of a model with a demand reaction to see which metabolites can not be produced
    store_dict = {}
    demand_dict = {}
    biomass_mets = [met.id for met in model.reactions.Growth.reactants]
    for met_id in biomass_mets:
        with model as model:
            # create and add synthetic reaction
            test_r = artificial_cytoplasm_addition(model, met_id)
            model.add_reactions([test_r])
            
            print(f"Added Reaction: {model.reactions.get_by_id(test_r.id).build_reaction_string()}")
            
            # optimize model 
            sol = model.optimize()
            print(f"Optimization Status: {sol.status}")
            print(f"Objective Value: {sol.objective_value}") 
            
            current_demand = demand_reaction(model, biomass_mets)
            demand_dict[met_id] = current_demand  
            ng_mets = get_non_growing_mets(current_demand)
            store_dict[met_id] = [ng_mets, sol.status, sol.objective_value]
    return store_dict, demand_dict

In [11]:
def compare_pre_postcur(prec_m, postc_m, met_id): #check fluxes for metabolites of interest pre and post curation steps 
    post_r = []
    pre_r = []
    pre_rxn = prec_m.metabolites.get_by_id(met_id).reactions
    post_rxn = postc_m.metabolites.get_by_id(met_id).reactions

    for rxn in pre_rxn:
        pre_r.append(rxn.id)
    for rxn in post_rxn:
        post_r.append(rxn.id)

    shared_rxn_ids = list(set(pre_r) & set(post_r))
    added_rxns = list(set(post_r) - set(pre_r))
    removed_rxns = list(set(pre_r) - set(post_r))

    print(f"Pre: {len(pre_rxn)}, Post: {len(post_rxn)}, Shared: {len(shared_rxn_ids)}")
    if added_rxns:
        print(f"Added in curation:    {added_rxns}")
    if removed_rxns:
        print(f"Removed in curation:  {removed_rxns}")

    # Check if bounds of shared reactions changed
    pre_bounds = {rxn.id: rxn.bounds for rxn in pre_rxn if rxn.id in shared_rxn_ids}
    post_bounds = {rxn.id: rxn.bounds for rxn in post_rxn if rxn.id in shared_rxn_ids}

    changed_bounds = {
        rxn_id: (pre_bounds[rxn_id], post_bounds[rxn_id])
        for rxn_id in shared_rxn_ids
        if pre_bounds[rxn_id] != post_bounds[rxn_id]
    }
    if changed_bounds:
        print("Reactions with changed bounds:")
        for rxn_id, (pre_b, post_b) in changed_bounds.items():
            print(f"  {rxn_id}: {pre_b} -> {post_b}")

    return pre_r, post_r

##### check draft models

In [46]:
all_blocked_draft = {}

for file in os.listdir(draft_model_dir):
    if file.endswith('.xml'):
        model = read_sbml_model(os.path.join(draft_model_dir,file))
        blocked_components, blocked_reactions = check_growth(model)
        all_blocked_draft.update(blocked_reactions)

Adding exchange reaction EX_3ump_e with default bounds for boundary metabolite: 3ump_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_ala_leu_e with default bounds for boundary metabolite: ala_leu_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary metabolite: 

Found 0 biomass components that cannot be synthesized for model m_2862.


Ignoring reaction 'EX_co2_e' since it already exists.
Ignoring reaction 'EX_co_e' since it already exists.
Ignoring reaction 'EX_coa_e' since it already exists.
Ignoring reaction 'EX_cobalt2_e' since it already exists.
Ignoring reaction 'EX_cu2_e' since it already exists.
Ignoring reaction 'EX_cyst__L_e' since it already exists.
Ignoring reaction 'EX_dtmp_e' since it already exists.
Ignoring reaction 'EX_fad_e' since it already exists.
Ignoring reaction 'EX_fald_e' since it already exists.
Ignoring reaction 'EX_fe2_e' since it already exists.
Ignoring reaction 'EX_fe3_e' since it already exists.
Ignoring reaction 'EX_fe3pyovd_kt_e' since it already exists.
Ignoring reaction 'EX_fol_e' since it already exists.
Ignoring reaction 'EX_gln__L_e' since it already exists.
Ignoring reaction 'EX_glyc3p_e' since it already exists.
Ignoring reaction 'EX_gmp_e' since it already exists.
Ignoring reaction 'EX_h2o_e' since it already exists.
Ignoring reaction 'EX_h2s_e' since it already exists.
Ignor

Found 0 biomass components that cannot be synthesized for model m_796.


Ignoring reaction 'EX_LalaDgluMdap_e' since it already exists.
Ignoring reaction 'EX_ac_e' since it already exists.
Ignoring reaction 'EX_acald_e' since it already exists.
Ignoring reaction 'EX_acgam1p_e' since it already exists.
Ignoring reaction 'EX_acnam_e' since it already exists.
Ignoring reaction 'EX_akg_e' since it already exists.
Ignoring reaction 'EX_alaala_e' since it already exists.
Ignoring reaction 'EX_amp_e' since it already exists.
Ignoring reaction 'EX_arg__L_e' since it already exists.
Ignoring reaction 'EX_argp_e' since it already exists.
Ignoring reaction 'EX_asn__L_e' since it already exists.
Ignoring reaction 'EX_bz_e' since it already exists.
Ignoring reaction 'EX_ca2_e' since it already exists.
Ignoring reaction 'EX_cl_e' since it already exists.
Ignoring reaction 'EX_co2_e' since it already exists.
Ignoring reaction 'EX_coa_e' since it already exists.
Ignoring reaction 'EX_cobalt2_e' since it already exists.
Ignoring reaction 'EX_cu2_e' since it already exists.


...

Found 0 biomass components that cannot be synthesized for model m_1234.


Ignoring reaction 'EX_2obut_e' since it already exists.
Ignoring reaction 'EX_5oxpro_e' since it already exists.
Ignoring reaction 'EX_6pgc_e' since it already exists.
Ignoring reaction 'EX_LalaDgluMdap_e' since it already exists.
Ignoring reaction 'EX_R_3httdca_e' since it already exists.
Ignoring reaction 'EX_ac_e' since it already exists.
Ignoring reaction 'EX_acac_e' since it already exists.
Ignoring reaction 'EX_acald_e' since it already exists.
Ignoring reaction 'EX_acgam1p_e' since it already exists.
Ignoring reaction 'EX_acmana_e' since it already exists.
Ignoring reaction 'EX_acnam_e' since it already exists.
Ignoring reaction 'EX_ala__L_e' since it already exists.
Ignoring reaction 'EX_alaala_e' since it already exists.
Ignoring reaction 'EX_amp_e' since it already exists.
Ignoring reaction 'EX_arg__L_e' since it already exists.
Ignoring reaction 'EX_asn__L_e' since it already exists.
Ignoring reaction 'EX_asp__L_e' since it already exists.
Ignoring reaction 'EX_bz_e' since i

Found 0 biomass components that cannot be synthesized for model m_761.
Found 0 biomass components that cannot be synthesized for model m_638.


Adding exchange reaction EX_2obut_e with default bounds for boundary metabolite: 2obut_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolit

Found 0 biomass components that cannot be synthesized for model m_939.


##### check curated models

In [15]:
all_blocked = {}
for file in os.listdir(model_dir):
    if file.endswith('_bz.xml'):
        model = read_sbml_model(os.path.join(model_dir,file))
        blocked_components, blocked_reactions = check_growth(model)
        all_blocked.update(blocked_reactions)

# growth was checked for all non growing models after each curation step to identify critical changes. 

Found 0 biomass components that cannot be synthesized for model m_790_.
Found 0 biomass components that cannot be synthesized for model m_638_.
Found 0 biomass components that cannot be synthesized for model m_761_.
Found 0 biomass components that cannot be synthesized for model m_793_.
Found 0 biomass components that cannot be synthesized for model m_504_.
Found 0 biomass components that cannot be synthesized for model m_161_.
Found 0 biomass components that cannot be synthesized for model m_1018_.
Found 0 biomass components that cannot be synthesized for model m_644_.
Found 0 biomass components that cannot be synthesized for model m_100_.
Found 0 biomass components that cannot be synthesized for model m_1234_.
Found 0 biomass components that cannot be synthesized for model m_778_.
Found 0 biomass components that cannot be synthesized for model m_352_.
Found 0 biomass components that cannot be synthesized for model m_1432_.
Found 0 biomass components that cannot be synthesized for mod

In [16]:
# store blocked metabolites in a file for a better overview 
metabolites_blocked_in = defaultdict(list)

for mod_id, met_list in all_blocked.items():
    for metabolite in met_list:
        metabolites_blocked_in[metabolite].append(mod_id)

store_blocked = pd.DataFrame.from_dict(metabolites_blocked_in, orient='index')

#store_blocked.columns = [f"Model_{i+1}" for i in range(store_blocked.shape[1])]
#filepath = os.path.join(working_dir, 'blocked_biomass_reactions_1006_1517.csv')

#store_blocked.to_csv(filepath)

In [17]:
store_blocked

,0
mql8_c,m_895_


#### Fe2+ Import: 428 and 644

##### diagnosis

In [30]:
pc428 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/428.xml") #pre curation 
pmb428 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/428_or_mb1.xml") #post mass balance

pc644 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/644.xml")
pmb644 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/644_or_mb1.xml") #post mass balance


Adding exchange reaction EX_6apa_e with default bounds for boundary metabolite: 6apa_e.
Adding exchange reaction EX_6pgc_e with default bounds for boundary metabolite: 6pgc_e.
Adding exchange reaction EX_LalaDgluMdap_e with default bounds for boundary metabolite: LalaDgluMdap_e.
Adding exchange reaction EX_R_3httdca_e with default bounds for boundary metabolite: R_3httdca_e.
Adding exchange reaction EX_ac_e with default bounds for boundary metabolite: ac_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_acnam_e with default bounds for boundary metabolite: acnam_e.
Adding exchange reaction EX_ala__L_e with default bounds for boundary metabolite: ala__L_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp

In [31]:
sol1 = pc428.optimize()
print(sol1.objective_value) 
sol2 = pmb428.optimize()
print(sol2.objective_value) #problem is post mass balance

16.876585114270682
0.0


In [32]:
sol1 = pc644.optimize()
print(sol1.objective_value) 
sol2 = pmb644.optimize()
print(sol2.objective_value) #problem is post mass balance

16.876585114270682
0.0


In [ ]:
# get all biomass precursors
biomass_prec_428 = [met.id for met in pc428.reactions.Growth.reactants]
biomass_prec_644 = [met.id for met in pc644.reactions.Growth.reactants]

In [ ]:
store_dict_428, demand_all_428 = test_biomass_prec(pmb428) #model has problem with Fe2+ import

Added Reaction:  --> 5.0 10fthf_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ala__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 amet_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 arg__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asn__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 asp__L_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 atp_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ca2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cl_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 coa_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 cobalt2_c
Optimization Status: optimal
Objective Value: 0.0
Added Reaction:  --> 5.0 ctp_c
Optimization Status: optimal
Objective Value: 0.0
Added Rea

In [ ]:
store_dict_644, demand_all_644 = test_biomass_prec(pmb644) #model has problem with Fe2+ import

##### Fix Fe2+ Import

In [18]:
def add_EX_fe2_e(model):
    add_new_rxn(model, "EX_fe2_e", "Exchange Reaction for Iron (Fe2+)", -1000, 1000,{"fe2_e": -1.0} )

In [19]:
def add_FE2tex(model):
    add_new_rxn(model, "FE2tex", "Transport Fe2 e -> p", -1000, 1000, {"fe2_e": -1.0, "fe2_p": 1.0} )

In [35]:
# fe2_p -> fe2_c FE2abcpp
def add_FE2abcpp(model):
    add_new_rxn(model, "FE2abcpp", "Transport Fe2 p->c", -1000, 1000, {"fe2_p": -1.0, 
                          "atp_c": -1.0, 
                          "h2o_c": -1.0, 
                          "adp_c": 1.0, 
                          "h_c": 1.0, 
                          "pi_c": 1.0,
                        "fe2_c": 1.0} )

In [36]:
fe2_models = [pmb428, pmb644]

In [38]:
for model in fe2_models: #double check if models can grow after Fe2+ pathway addition
    with model:
        print(model.id.split("_")[1])
        sol1 = model.optimize()
        print("pre iron addition", sol1.objective_value, sol1.status)
        add_new_met(model, "fe2_e", "Iron (Fe2+)", "Fe", 2, "C_e")
        add_new_met(model, "fe2_p", "Iron (Fe2+)", "Fe", 2, "C_p")
        add_EX_fe2_e(model)
        add_FE2tex(model)
        add_FE2abcpp(model)
        sol2 = model.optimize()
        print("post iron import", sol2.objective_value, sol2.status)
        write_sbml_model(model, mb_dir+f'{model.id.split("_")[1]}_or_mb1.xml') #save overwritten models

428
pre iron addition 0.0 optimal
post iron import 16.87658511427081 optimal
644
pre iron addition 0.0 optimal
post iron import 16.87658511427081 optimal


#### Reversibility of reactions: example 2751

In [11]:
pcm2751 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models/2751.xml")
pmb2751 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/2751_or_mb1.xml") #post mass balance
pmc2751 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_lib/2751_or_mb1_mdr_rdr_dp_lib.xml") #post macaw

Adding exchange reaction EX_LalaDgluMdapDala_e with default bounds for boundary metabolite: LalaDgluMdapDala_e.
Adding exchange reaction EX_acald_e with default bounds for boundary metabolite: acald_e.
Adding exchange reaction EX_acgam1p_e with default bounds for boundary metabolite: acgam1p_e.
Adding exchange reaction EX_acmana_e with default bounds for boundary metabolite: acmana_e.
Adding exchange reaction EX_akg_e with default bounds for boundary metabolite: akg_e.
Adding exchange reaction EX_alaala_e with default bounds for boundary metabolite: alaala_e.
Adding exchange reaction EX_alagly_e with default bounds for boundary metabolite: alagly_e.
Adding exchange reaction EX_amp_e with default bounds for boundary metabolite: amp_e.
Adding exchange reaction EX_arg__L_e with default bounds for boundary metabolite: arg__L_e.
Adding exchange reaction EX_asn__L_e with default bounds for boundary metabolite: asn__L_e.
Adding exchange reaction EX_asp__D_e with default bounds for boundary me

In [ ]:
biomass_prec_2751 = [met.id for met in pcm2751.reactions.Growth.reactants]

In [ ]:
for met_id in biomass_prec_2751:
    compare_pre_postcur(pcm2751, pmc2751, met_id)

Pre: 2, Post: 2, Shared: 2
Pre: 4, Post: 4, Shared: 4
Pre: 3, Post: 3, Shared: 3
Pre: 3, Post: 3, Shared: 3
Pre: 3, Post: 3, Shared: 3
Pre: 5, Post: 5, Shared: 5
Pre: 44, Post: 44, Shared: 43
Added in curation:    ['RBFK']
Removed in curation:  ['RBFK_1']
Reactions with changed bounds:
  NNATr: (-1000.0, 1000.0) -> (0.0, 1000.0)
  OXACOAL: (-1000.0, 1000.0) -> (0.0, 1000.0)
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 12, Post: 12, Shared: 12
Reactions with changed bounds:
  OXACOAL: (-1000.0, 1000.0) -> (0.0, 1000.0)
Pre: 2, Post: 2, Shared: 2
Pre: 3, Post: 3, Shared: 3
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 2, Post: 2, Shared: 2
Pre: 3, Post: 3, Shared: 3
Pre: 5, Post: 5, Shared: 5
Pre: 3, Post: 3, Shared: 3
Pre: 4, Post: 4, Shared: 4
Pre: 8, Post: 8, Shared: 8
Pre: 7, Post: 7, Shared: 7
Pre: 3, Post: 3, Shared: 3
Pre: 3, Post: 3, Shared: 3
Pre: 46, Post: 46, Shared: 46
Pre: 2, Post: 2, Shared: 2


In [ ]:
# the cell above identifies three reactions with changed bounds/altered reversibility - now I need to check if changing the reversibility back to the draft stage, leads to growth 
make_rev = ["OXACOAL", "NAPRT", "NNATr"] 
with pmc2751 as model:
    for rxn_id in make_rev:
        sol = model.optimize()
        print(rxn_id, sol.objective_value)
        model.reactions.get_by_id(rxn_id).bounds = (-1000, 1000)
        sol = model.optimize()
        print(rxn_id, sol.objective_value)

# I need to change the reversibility of OXACOAL -> add check (function check_influence_reversibility) to 06_MACAW_fixes 

OXACOAL -1.9067966267607355e-15
OXACOAL 16.772360503419844
NAPRT 16.772360503419826
NAPRT 16.772360503419826
NNATr 16.772360503419826
NNATr 16.772360503419826
